# PTB-XL Pretraining for Mitral ECG Backbone

Pretrain the `ECGResNet1D` backbone on PTB-XL before fine-tuning on the mitral classification task.

## PTB-XL on Kaggle

Add this dataset to your Kaggle notebook:
```
ptb-xl-a-large-publicly-available-electrocardiography-dataset
```
It will appear at `/kaggle/input/ptb-xl-a-large-publicly-available-electrocardiography-dataset/`.

Install wfdb if not already present: `!pip install wfdb -q`

In [ ]:
from __future__ import annotations

import ast
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

try:
    import wfdb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wfdb", "-q"])
    import wfdb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PTBXL_DIR    = Path("/kaggle/input/datasets/sidalikhelilcherfi/ptbxl-ecg-database")
OUT_DIR      = Path(".") / "dl_artifacts" / "ptbxl_pretrain"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE   = 64
EPOCHS       = 60
PATIENCE     = 12
LR           = 1e-3
MIN_LR       = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 5.0
NUM_WORKERS  = 0       # 0 = main process; set higher if not preloading to RAM
PRELOAD_TO_RAM   = True
PRELOAD_WORKERS  = 4

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

print("PTB-XL dir exists:", PTBXL_DIR.exists())
print("Output dir:", OUT_DIR)
print("Device:", DEVICE)

## 1. Load PTB-XL metadata and build labels

In [ ]:
db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)

# load the SCP statement table to get superclass mappings
scp_stmts = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
superclass_map = scp_stmts[scp_stmts["diagnostic"] == 1]["diagnostic_class"].dropna().to_dict()

def get_superclass_labels(scp_codes: dict) -> list[int]:
    """Return multi-hot vector over SUPERCLASSES for a sample."""
    present = set()
    for code, likelihood in scp_codes.items():
        if likelihood >= 50 and code in superclass_map:
            present.add(superclass_map[code])
    return [int(sc in present) for sc in SUPERCLASSES]

db["label_vec"] = db["scp_codes"].apply(get_superclass_labels)

# keep only samples that have at least one superclass label
db = db[db["label_vec"].apply(sum) > 0].copy()

# train/val split using the official strat_fold column
# folds 1-8 = train, fold 9 = val, fold 10 = test (standard PTB-XL protocol)
train_db = db[db["strat_fold"] <= 8].copy()
val_db   = db[db["strat_fold"] == 9].copy()

print(f"Train: {len(train_db)}, Val: {len(val_db)}")
print("Superclass distribution (train):")
label_mat = np.stack(train_db["label_vec"].values)
for i, sc in enumerate(SUPERCLASSES):
    print(f"  {sc}: {label_mat[:, i].sum()} ({label_mat[:, i].mean()*100:.1f}%)")

## 2. Dataset

In [ ]:
PTBXL_RAM_CACHE: dict[str, np.ndarray] = {}  # stem -> (36, 5000) float32


def load_ptbxl_signal(record_path: str) -> np.ndarray:
    """Load one PTB-XL record and return a (36, 5000) float32 array.
    Channels 0-11: normalised signal. Channels 12-35: zeros (mask placeholder).
    """
    signal, _ = wfdb.rdsamp(record_path)
    signal = normalize_per_sample_per_lead(signal.astype(np.float32))
    x = np.zeros((36, 5000), dtype=np.float32)
    x[:12] = signal.T
    return x


def preload_ptbxl(frame: pd.DataFrame, ptbxl_dir: Path, desc: str) -> None:
    if not PRELOAD_TO_RAM:
        return
    pending = [row for row in frame[["ecg_id", "filename_hr"]].to_dict("records")
               if str(row["ecg_id"]) not in PTBXL_RAM_CACHE]
    before = len(PTBXL_RAM_CACHE)

    def load_one(row: dict) -> tuple[str, np.ndarray]:
        path = str(ptbxl_dir / row["filename_hr"])
        return str(row["ecg_id"]), load_ptbxl_signal(path)

    if pending:
        with ThreadPoolExecutor(max_workers=PRELOAD_WORKERS) as ex:
            futures = [ex.submit(load_one, row) for row in pending]
            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"preload {desc}", leave=False):
                ecg_id, arr = fut.result()
                PTBXL_RAM_CACHE[ecg_id] = arr

    added = len(PTBXL_RAM_CACHE) - before
    total_gb = sum(a.nbytes for a in PTBXL_RAM_CACHE.values()) / 1024**3
    print(f"Preloaded {added} new PTB-XL records for {desc}; "
          f"cache={len(PTBXL_RAM_CACHE)}, RAM≈{total_gb:.1f} GB")


def normalize_per_sample_per_lead(signal: np.ndarray) -> np.ndarray:
    """Same normalization used in the mitral fine-tuning pipeline."""
    mu  = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True).clip(min=1e-6)
    return ((signal - mu) / std).astype(np.float32)


def augment(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Light augmentation on signal channels only (mask channels stay zero)."""
    n_signal = 12
    if rng.random() < 0.5:
        x[:n_signal] *= rng.uniform(0.95, 1.05, size=(n_signal, 1)).astype(np.float32)
    if rng.random() < 0.5:
        x[:n_signal] += rng.normal(0.0, 0.01, size=(n_signal, x.shape[1])).astype(np.float32)
    if rng.random() < 0.35:
        n_steps = x.shape[1]
        width = int(rng.integers(1, max(2, int(n_steps * 0.03) + 1)))
        start = int(rng.integers(0, max(1, n_steps - width + 1)))
        x[:n_signal, start:start + width] = 0.0
    return x


class PTBXLDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, ptbxl_dir: Path,
                 augment_: bool = False, seed: int = SEED):
        self.frame    = frame.reset_index()
        self.dir      = ptbxl_dir
        self.augment_ = augment_
        self.seed     = seed
        preload_ptbxl(self.frame, ptbxl_dir, desc="train" if augment_ else "val")

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        ecg_id = str(row["ecg_id"])

        if PRELOAD_TO_RAM and ecg_id in PTBXL_RAM_CACHE:
            x = PTBXL_RAM_CACHE[ecg_id].copy()
        else:
            x = load_ptbxl_signal(str(self.dir / row["filename_hr"]))

        if self.augment_:
            rng = np.random.default_rng(self.seed + idx * 1009 + random.randint(0, 1_000_000))
            x = augment(x, rng)

        y = torch.tensor(row["label_vec"], dtype=torch.float32)
        return torch.from_numpy(x), y


train_ds = PTBXLDataset(train_db, PTBXL_DIR, augment_=True)
val_ds   = PTBXLDataset(val_db,   PTBXL_DIR, augment_=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

x_sample, y_sample = train_ds[0]
print("Sample x shape:", x_sample.shape)          # (36, 5000)
print("Mask channels are zero:", (x_sample[12:] == 0).all().item())
print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

## 3. Model — same backbone, 12-channel input, 5-class multi-label head

In [ ]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1,
                 kernel_size: int = 17, dropout: float = 0.0):
        super().__init__()
        padding = kernel_size // 2
        self.conv1      = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False)
        self.bn1        = nn.BatchNorm1d(out_channels)
        self.act        = nn.ReLU(inplace=True)
        self.dropout    = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2      = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, bias=False)
        self.bn2        = nn.BatchNorm1d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.act(out + identity)


class ECGResNet1D(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, dropout: float = 0.30):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=17, padding=8, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            ResidualBlock1D(64,  64,  1, 17, 0.00), ResidualBlock1D(64,  64,  1, 17, 0.00),
            ResidualBlock1D(64,  128, 2, 17, 0.05), ResidualBlock1D(128, 128, 1, 17, 0.05),
            ResidualBlock1D(128, 196, 2, 17, 0.10), ResidualBlock1D(196, 196, 1, 17, 0.10),
            ResidualBlock1D(196, 256, 2, 17, 0.15), ResidualBlock1D(256, 256, 1, 17, 0.15),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.blocks(self.stem(x))))

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        return self.pool(self.blocks(self.stem(x))).squeeze(-1)


# 36 channels during pretraining — mask channels zero-padded
model = ECGResNet1D(in_channels=36, num_classes=len(SUPERCLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(f"SUPERCLASSES: {SUPERCLASSES}")
print("Input channels: 36 (12 signal + 24 zero-padded mask channels)")

## 4. Pretraining loop

In [ ]:
# positive class weights to handle PTB-XL imbalance
label_mat  = np.stack(train_db["label_vec"].values).astype(float)
pos_weight = torch.tensor(
    (1 - label_mat.mean(axis=0)) / label_mat.mean(axis=0).clip(min=1e-6),
    dtype=torch.float32,
).to(DEVICE)
print("pos_weight:", pos_weight.cpu().numpy().round(2))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)


def run_epoch(loader, train: bool) -> tuple[float, float]:
    model.train(train)
    total_loss = 0.0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False, desc="train" if train else "val"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss   = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            total_loss += loss.item() * len(x)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(y.cpu().numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    try:
        auc = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        auc = float("nan")
    return total_loss / len(loader.dataset), auc


best_val_auc  = -1.0
patience_ctr  = 0
history       = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_auc = run_epoch(train_loader, train=True)
    val_loss,   val_auc   = run_epoch(val_loader,   train=False)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    history.append({"epoch": epoch, "train_loss": train_loss, "train_auc": train_auc,
                    "val_loss": val_loss, "val_auc": val_auc, "lr": lr_now})
    print(f"E{epoch:03d}  train_loss={train_loss:.4f} train_auc={train_auc:.3f}  "
          f"val_loss={val_loss:.4f} val_auc={val_auc:.3f}  lr={lr_now:.2e}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_ctr = 0
        torch.save(model.state_dict(), OUT_DIR / "ptbxl_pretrained_best.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

pd.DataFrame(history).to_csv(OUT_DIR / "pretrain_history.csv", index=False)
print(f"\nBest val AUC: {best_val_auc:.4f}")
print("Saved:", OUT_DIR / "ptbxl_pretrained_best.pt")

## 5. Training curve

In [ ]:
import matplotlib.pyplot as plt

hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["epoch"], hist["train_loss"], label="train")
axes[0].plot(hist["epoch"], hist["val_loss"],   label="val")
axes[0].set_title("BCE loss")
axes[0].legend()
axes[1].plot(hist["epoch"], hist["train_auc"], label="train")
axes[1].plot(hist["epoch"], hist["val_auc"],   label="val")
axes[1].set_title("Macro OVR AUC")
axes[1].legend()
plt.suptitle("PTB-XL pretraining")
plt.tight_layout()
plt.savefig(OUT_DIR / "pretrain_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Save backbone weights

Save `stem`, `blocks`, and `pool` — everything except the final linear classifier.
Since pretraining and fine-tuning both use 36 input channels, the **stem can be fully transferred**.
The mask-channel filters in the stem stayed near their random initialization (zero gradient during pretraining),
but the signal-channel filters are pretrained. Fine-tuning teaches the mask filters from there.

In [ ]:
# reload best checkpoint
model.load_state_dict(torch.load(OUT_DIR / "ptbxl_pretrained_best.pt", map_location="cpu"))

# extract everything except the final linear layer
backbone_state = {
    k: v for k, v in model.state_dict().items()
    if not k.startswith("head.2")  # head.2 is the Linear(256, 5) layer
}
torch.save(backbone_state, OUT_DIR / "ptbxl_backbone_weights.pt")

print("Backbone keys saved:", len(backbone_state))
print("Keys include stem:", any(k.startswith("stem") for k in backbone_state))
print("Keys include blocks:", any(k.startswith("blocks") for k in backbone_state))
print("Saved:", OUT_DIR / "ptbxl_backbone_weights.pt")